Ячейка 2 — Setup

In [1]:
# %%
"""
SETUP
"""
import json
import re
from pathlib import Path
from datetime import datetime

import pandas as pd

STRUCT = Path("../data/structured_data")
OUT_DIR = Path("../data/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BOSSES_PATH = STRUCT / "bosses" / "all.json"
UVH_PATH = STRUCT / "bosses" / "uvh_locations.json"
DICT_PATH = Path("../data/dictionary/result/dictionary_en_ru.json")

MISSING = "-"

def load(p: Path):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", (s or "").lower())

def dash(x):
    if x is None or x == "" or x == []:
        return MISSING
    if isinstance(x, bool):
        return "yes" if x else "no"
    return x

print("STRUCT:", STRUCT.resolve())
print("OUT:   ", OUT_DIR.resolve())
print("DICT:  ", DICT_PATH.resolve(), "| exists:", DICT_PATH.exists())

STRUCT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/structured_data
OUT:    /var/home/nexpg/RawBL4ToCutBDInterpreter/data/output
DICT:   /var/home/nexpg/RawBL4ToCutBDInterpreter/data/dictionary/result/dictionary_en_ru.json | exists: True


Ячейка 3 — Load structured

In [2]:
# %%
"""
LOAD structured bosses + dictionary (guid → ru)
"""
if not BOSSES_PATH.exists():
    raise FileNotFoundError(f"Нет {BOSSES_PATH} — сначала proгони raw_ncs_to_structured")

bosses = load(BOSSES_PATH)
uvh_rows = load(UVH_PATH) if UVH_PATH.exists() else []

print(f"bosses: {len(bosses)}")
print(f"uvh rows: {len(uvh_rows)}")

# --- dictionary ---
guid_map: dict[str, dict] = {}
en_lower_map: dict[str, list[str]] = {}

if DICT_PATH.exists():
    raw_dict = load(DICT_PATH)
    for k, v in raw_dict.items():
        if not isinstance(v, dict):
            continue
        entry = {
            "en": (v.get("en") or "").strip(),
            "ru": (v.get("ru") or "").strip(),
        }
        ku = str(k).replace("-", "").upper()
        guid_map[ku] = entry
        guid_map[str(k).lower()] = entry
        en = entry["en"].lower()
        if en and entry["ru"]:
            en_lower_map.setdefault(en, [])
            if entry["ru"] not in en_lower_map[en]:
                en_lower_map[en].append(entry["ru"])
    print(f"dictionary guids: {len(guid_map)} | en keys: {len(en_lower_map)}")
else:
    print("WARNING: dictionary not found — display_name_ru will be '-' or '(перевод не найден)'")

def translate_ru(guid: str | None, en_text: str | None) -> str:
    """
    1) по GUID
    2) по точному en (lower)
    иначе '(перевод не найден)' если en есть, иначе MISSING
    несколько вариантов → '(требуется ручная проверка)'
    """
    if guid:
        g = str(guid).replace("-", "").upper()
        hit = guid_map.get(g) or guid_map.get(str(guid).lower())
        if hit:
            ru = (hit.get("ru") or "").strip()
            if ru:
                return ru

    if en_text:
        variants = en_lower_map.get(en_text.strip().lower(), [])
        variants = [v for v in variants if v and v.strip()]
        if len(variants) == 1:
            return variants[0]
        if len(variants) > 1:
            return "(требуется ручная проверка)"
        return "(перевод не найден)"

    return MISSING

bosses: 82
uvh rows: 21
dictionary guids: 232656 | en keys: 61228


Ячейка 4 — Resolve display_name + region

In [3]:
# %%
"""
BUILD ROWS
display_name / region / handles — только из structured (Pass I).
display_name_ru — словарь по display_guid, иначе по en-фразе.
Никаких DISPLAY_MAP в коде.
"""
rows = []
stats = {
    "from_structured_name": 0,
    "no_name": 0,
    "with_region": 0,
    "ru_ok": 0,
    "ru_missing": 0,
}

for b in bosses:
    key = (b.get("boss_key") or "").strip()
    if not key:
        continue

    # --- из structured (NCS) ---
    display_name = b.get("display_name")  # str | None
    display_guid = b.get("display_guid")
    region = b.get("region")
    has_trueboss = bool(b.get("has_trueboss"))
    has_true = bool(b.get("has_true"))
    handles = b.get("dedicated_handles") or []
    handles_str = "; ".join(handles) if handles else None

    if display_name:
        stats["from_structured_name"] += 1
    else:
        stats["no_name"] += 1

    if region:
        stats["with_region"] += 1

    # --- ru ---
    if display_name:
        name_ru = translate_ru(display_guid, display_name)
        if name_ru not in (MISSING, "(перевод не найден)", "(требуется ручная проверка)"):
            stats["ru_ok"] += 1
        else:
            stats["ru_missing"] += 1
    else:
        name_ru = MISSING

    rows.append({
        "boss_key": key,
        "display_name": dash(display_name),
        "display_name_ru": dash(name_ru) if name_ru != MISSING else name_ru,
        # если name_ru уже спец-строка — оставляем как есть
        "has_trueboss": dash(has_trueboss),
        "has_true": dash(has_true),
        "region": dash(region),
        "dedicated_handles": dash(handles_str),
    })

# поправить display_name_ru: спец-метки не через dash
for r in rows:
    ru = r["display_name_ru"]
    if ru in ("(перевод не найден)", "(требуется ручная проверка)"):
        pass  # already set
    elif r["display_name"] == MISSING:
        r["display_name_ru"] = MISSING

print("stats:", stats)
print("rows:", len(rows))

stats: {'from_structured_name': 53, 'no_name': 29, 'with_region': 9, 'ru_ok': 53, 'ru_missing': 0}
rows: 82


Ячейка 5 — DataFrame + preview

In [4]:
# %%
"""
DATAFRAME
"""
COLS = [
    "boss_key",
    "display_name",
    "display_name_ru",
    "has_trueboss",
    "has_true",
    "region",
    "dedicated_handles",
]

df = pd.DataFrame(rows, columns=COLS)
df = df.sort_values("boss_key").reset_index(drop=True)

print(df.head(25).to_string(index=False))
print()
print(
    "named:", (df["display_name"] != MISSING).sum(),
    "| ru:", (df["display_name_ru"] != MISSING).sum(),
    "| region:", (df["region"] != MISSING).sum(),
    "| handles:", (df["dedicated_handles"] != MISSING).sum(),
)

                         boss_key                                                         display_name                                              display_name_ru has_trueboss has_true         region                                                                                                                                                                       dedicated_handles
                            arjay                                                                Arjay                                                       Арджей          yes       no              -                                                                                 dad_sg.comp_05_legendary_heartgun; dad_shield.comp_05_legendary_angel; ord_sr.comp_05_legendary_fisheye
                         backhive                                                             Backhive                                                     Ульеспин          yes       no              -                              

Ячейка 6 — Save CSV

In [5]:
# %%
"""
SAVE CSV
имя: bl4_bosses_MM-DD-YYYY_HH-MM-SS.csv  (американский формат)
"""
now = datetime.now()
stamp = now.strftime("%m-%d-%Y_%H-%M-%S")
out_path = OUT_DIR / f"bl4_bosses_{stamp}.csv"

df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path.resolve())
print("shape:", df.shape)

Saved: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/output/bl4_bosses_08-14-2026_23-12-31.csv
shape: (82, 7)
